# A Minimal Agent Harness

**The harness is the agent. The LLM is a function it calls.**

This notebook builds, box by box, the harness from our diagram:

`TASK/GOAL → observe → goal achieved? → compose prompt → LLM → process response → consult H → act (via tools) → back to observe`

Everything inside the green box is ordinary Python, about 150 lines. The LLM and the human (H) sit outside it.

## Roadmap: one function per box

| Diagram box | Function in this notebook |
|---|---|
| WORLD | `world` (a plain Python dict) |
| TASK/GOAL + "goal achieved?" | `GOAL`, `goal_achieved()` |
| tool portfolio | `TOOLS` |
| observe | `observe()` |
| history | `history` (a list) |
| compose prompt | `compose_prompt()` |
| LLM | `scripted_llm`, `human_llm`, `claude_llm` (interchangeable) |
| process response | `process_response()` |
| consult (H) | `policy_consult`, `human_consult` |
| act | `act()` |
| the loop itself | `run_agent()` |

**Strategy to watch for:** each box is a separate function with a narrow job, so we can swap any one of them (a different LLM, a stricter consult policy) without touching the others. Real harnesses are built the same way.

In [ ]:
import json, re, textwrap

def say(box, text):
    # Print one labelled line per diagram box so the run is easy to follow.
    print(f"[{box:>16}] " + textwrap.shorten(str(text), width=110, placeholder=" …"))

## WORLD

The world is just a dictionary: some files and an email outbox.

**Key idea:** nothing in the harness touches `world` directly except the **tools** and the **goal check**. The LLM never sees the world; it sees only what the harness chooses to show it.

In [ ]:
def make_world():
    return {
        "files": {
            "notes.txt": ("Q3 project meeting: launch slipped two weeks because of a vendor delay; "
                          "budget is on track; next review is Nov 3."),
        },
        "outbox": [],
    }

world = make_world()
world

## TASK/GOAL and the "goal achieved?" diamond

The goal is plain text, because that's what goes into the prompt.

The *check* is code. Notice that `goal_achieved()` reads the **real world**, a privilege the LLM doesn't have. This is the "harness judges completion" design from the slides: it's a `while` loop, and it can stop before doing anything.

*Question for discussion:* for most real tasks ("write a good report"), could you write this function at all? If not, who decides when the agent is done?

In [ ]:
BOSS = "boss@example.com"

GOAL = ("Read notes.txt, write a one-sentence summary of it to report.txt, "
        f"and email that summary to {BOSS}.")

def goal_achieved(world):
    report = world["files"].get("report.txt", "").strip()
    emailed = any(m["to"] == BOSS and m["body"].strip() == report for m in world["outbox"])
    return bool(report) and emailed

goal_achieved(world)

## Tool portfolio

Tools are the **only** way the agent can affect the world. The list of tools *is* the agent's capability: no `delete_file` tool, no deletions.

Each tool has two parts:
- a **description**, which goes into the prompt so the model knows what it can do, and
- a **function**, which the harness runs. The model never runs anything itself.

`RISKY` marks the tools the consult step should look at more closely.

In [ ]:
def list_files(world):
    return "Files: " + ", ".join(sorted(world["files"])) if world["files"] else "No files."

def read_file(world, name):
    return world["files"].get(name, f"ERROR: no file named {name!r}")

def write_file(world, name, text):
    world["files"][name] = text
    return f"Wrote {len(text)} characters to {name}."

def delete_file(world, name):
    return f"Deleted {name}." if world["files"].pop(name, None) is not None else f"ERROR: no file {name!r}"

def send_email(world, to, body):
    world["outbox"].append({"to": to, "body": body})
    return f"Email sent to {to}."

TOOLS = {
    "list_files":  (list_files,  "list_files() - list the files that exist"),
    "read_file":   (read_file,   "read_file(name) - return the contents of a file"),
    "write_file":  (write_file,  "write_file(name, text) - create or overwrite a file"),
    "delete_file": (delete_file, "delete_file(name) - permanently delete a file"),
    "send_email":  (send_email,  "send_email(to, body) - send an email (cannot be unsent)"),
}

RISKY = {"write_file", "delete_file", "send_email"}

## observe

`observe()` turns the world into a short **text description**. Notice how much it leaves out: it lists file *names* but not their contents. To learn what's in a file, the agent has to use `read_file`.

Whatever this function omits, the agent can't perceive. Designing the observation is designing the agent's senses.

In [ ]:
def observe(world):
    files = ", ".join(sorted(world["files"])) or "(none)"
    return f"Files present: {files}. Emails sent so far: {len(world['outbox'])}."

observe(world)

## history and compose prompt

The LLM has **no memory between calls**. Each call starts from scratch. Its only "memory" is the `history` list that the harness keeps and pastes back into every prompt.

`compose_prompt()` assembles a single string from five parts: instructions and response format, the goal, the tool descriptions, the history, and the current observation.

**Everything the model "knows" about the task arrives through this one function.**

In [ ]:
def format_history(history):
    if not history:
        return "(nothing yet)"
    return "\n".join(f"step {h['step']} | {h['kind']}: {h['text']}" for h in history)

def compose_prompt(goal, observation, history):
    tool_list = "\n".join(f"- {desc}" for _, desc in TOOLS.values())
    return f'''You are an agent working toward a goal. Choose ONE next action.

GOAL: {goal}

TOOLS YOU CAN USE:
{tool_list}

HISTORY OF WHAT HAS HAPPENED:
{format_history(history)}

CURRENT OBSERVATION: {observation}

Reply with ONLY a JSON object, like:
{{"thought": "why this step", "tool": "read_file", "args": {{"name": "notes.txt"}}}}
If a human refused an action, take that into account and do something else.'''

print(compose_prompt(GOAL, observe(world), []))

## LLM: three interchangeable "minds"

To the harness, an LLM is just a **function from text to text**: `llm(prompt) -> str`. Anything with that shape will do:

1. **`scripted_llm`**: replays a fixed list of replies. It runs anywhere, with no API key, and does the same thing every time. It *ignores the prompt*, so it can't learn from a refusal.
2. **`human_llm`**: *you* play the LLM. You see exactly the prompt the model would see, and you type the JSON reply.
3. **`claude_llm`**: a real model, called through the Anthropic API.

That they're interchangeable is the point: the harness doesn't care what kind of mind is producing the text.

In [ ]:
def scripted_llm(replies):
    # Returns an llm function that hands back the next canned reply each time it's called.
    replies = iter(replies)
    def llm(prompt):
        return next(replies, '{"thought": "I have run out of ideas.", "tool": "list_files", "args": {}}')
    return llm

DEMO_SCRIPT = [
    '{"thought": "First, see what files exist.", "tool": "list_files", "args": {}}',
    'Sure! I will read the notes file now.',                       # not JSON: tests process_response
    '{"thought": "Read the notes.", "tool": "read_file", "args": {"name": "notes.txt"}}',
    '{"thought": "Write the summary.", "tool": "write_file", "args": {"name": "report.txt", '
        '"text": "Launch slipped two weeks due to a vendor delay; budget on track; next review Nov 3."}}',
    '{"thought": "Tidy up: the notes are no longer needed.", "tool": "delete_file", "args": {"name": "notes.txt"}}',
    '{"thought": "Email the summary to the boss.", "tool": "send_email", "args": {"to": "boss@example.com", '
        '"body": "Launch slipped two weeks due to a vendor delay; budget on track; next review Nov 3."}}',
]

In [ ]:
def human_llm(prompt):
    # You are the LLM: read the prompt, type a JSON reply.
    print("\n" + "=" * 30 + " PROMPT SENT TO THE 'LLM' (you) " + "=" * 30)
    print(prompt)
    print("=" * 92)
    return input("Your JSON reply: ")

In [ ]:
# Optional: a real model. In Colab, add your key under the key icon (Secrets) as ANTHROPIC_API_KEY.
MODEL = "claude-sonnet-5-5"

def get_api_key():
    import os
    try:
        from google.colab import userdata
        return userdata.get("ANTHROPIC_API_KEY")
    except Exception:
        return os.environ.get("ANTHROPIC_API_KEY")

def claude_llm(prompt):
    import anthropic                      # pip install anthropic, if needed
    client = anthropic.Anthropic(api_key=get_api_key())
    msg = client.messages.create(model=MODEL, max_tokens=400,
                                 messages=[{"role": "user", "content": prompt}])
    return msg.content[0].text

## process response

The LLM returns **text**, not an action. The harness has to turn that text into something it can run, and it has to cope when the text is malformed. Real models often add chatter or wrap their JSON in ``` fences.

If parsing fails, the harness doesn't crash. It records the error in history and goes around the loop again, so the model can see its mistake next time.

In [ ]:
def process_response(text):
    # Return {"thought", "tool", "args"} or None if the reply can't be used.
    match = re.search(r"\{.*\}", text, re.DOTALL)     # tolerate chatter or ```json fences
    if not match:
        return None
    try:
        action = json.loads(match.group())
    except json.JSONDecodeError:
        return None
    if action.get("tool") not in TOOLS:
        return None
    action.setdefault("args", {})
    action.setdefault("thought", "")
    return action

print(process_response('Here you go:\n```json\n{"tool": "list_files", "args": {}}\n```'))
print(process_response("Sure! I will read the notes file now."))

## consult (H)

Before acting, the harness asks someone (or something) outside itself for permission. Three possible answers:

- **`y`**: go ahead and act
- **`n`**: don't. The refusal goes into history, and the loop carries on, so the model can adapt.
- **`stop`**: end the whole run. A principal who can only veto single actions, and never halt the agent, doesn't have full oversight.

Note *who describes the action to H*. The harness shows the **actual tool and arguments**, and separately shows the model's stated reason, labelled as the model's claim. The reason isn't evidence; the arguments are what will actually run.

In [ ]:
def describe(action):
    return f"{action['tool']}({json.dumps(action['args'])})"

def policy_consult(action):
    # An automatic H: approve everything except deletions.
    return "n" if action["tool"] == "delete_file" else "y"

def human_consult(action):
    # A real H: safe tools are auto-approved, risky ones need a human decision.
    if action["tool"] not in RISKY:
        return "y"
    print(f"\n  The agent wants to run: {describe(action)}")
    print(f"  The model says why:     {action['thought']!r}")
    answer = ""
    while answer not in ("y", "n", "stop"):
        answer = input("  Approve? [y / n / stop]: ").strip().lower()
    return answer

## act

`act()` looks up the tool and runs it **against the world**. This is the only place the world changes. The tool's result (success or an error) goes into history, which is how the agent learns what its action did.

In [ ]:
def act(action, world):
    func, _ = TOOLS[action["tool"]]
    try:
        return func(world, **action["args"])
    except TypeError as e:                 # wrong arguments, e.g. a missing 'name'
        return f"ERROR: bad arguments for {action['tool']}: {e}"

## The loop: `run_agent()`

Here's the whole harness. Compare it line by line with the diagram.

Two code tactics are worth noticing:
- **`while True` with `return`** is how code expresses the flowchart's junction circle. We check *after* observing but *before* acting, which is the loop-and-a-half pattern.
- **`max_steps`** is a budget: a second way to reach "stop". Without it, a model that keeps proposing refused actions would loop forever.

In [ ]:
def run_agent(goal, llm, consult, world, max_steps=10):
    history = []
    step = 0
    while True:
        step += 1
        if step > max_steps:                                  # budget: second way to stop
            say("stop", f"Out of budget after {max_steps} steps.")
            return history

        observation = observe(world)                          # observe
        say("observe", observation)

        if goal_achieved(world):                              # task accomplished?
            say("stop", "Goal achieved.")
            return history

        prompt = compose_prompt(goal, observation, history)   # compose prompt
        reply = llm(prompt)                                   # LLM (outside the harness)
        say("LLM", reply)

        action = process_response(reply)                      # process response
        if action is None:
            say("process response", "Couldn't parse a valid action; noting it and retrying.")
            history.append({"step": step, "kind": "error", "text": f"Unusable reply: {reply!r}"})
            continue
        history.append({"step": step, "kind": "agent proposed",
                        "text": f"{describe(action)} because {action['thought']!r}"})

        decision = consult(action)                            # consult H
        say("consult H", f"{describe(action)} -> {decision}")
        if decision == "stop":
            history.append({"step": step, "kind": "human", "text": "STOPPED the run."})
            say("stop", "Human stopped the run.")
            return history
        if decision == "n":
            history.append({"step": step, "kind": "human", "text": f"REFUSED {describe(action)}"})
            continue                                          # refusal is in history; loop again

        result = act(action, world)                           # act (via tool portfolio)
        say("act", result)
        history.append({"step": step, "kind": "result", "text": result})

## Run 1: scripted LLM, automatic consult

The script includes two traps:
- one reply that isn't JSON, to exercise **process response**, and
- an unrequested "tidy up" deletion, to exercise **consult**.

In [ ]:
world = make_world()
history = run_agent(GOAL, scripted_llm(DEMO_SCRIPT), policy_consult, world)

## What the agent "remembers"

This is the history the model would see on its next call. Notice that the refusal is in it.

The scripted LLM never reads its prompt, so it couldn't have learned from the refusal anyway. It just carried on with its script. A real model *reads* the history, so recording the refusal is what lets it adapt.

In [ ]:
print(format_history(history))
print("\nFinal world:", json.dumps(world, indent=2))

## Run 2: you are the LLM

Set `PLAY = True` and run the cell. You'll see exactly the prompt a model would see, and you type the JSON reply.

Try:
- Finishing the task with as few steps as possible.
- Proposing `delete_file`, refusing it yourself at the consult step, and noticing that the refusal appears in your next prompt.
- Typing something that isn't JSON, and seeing what the harness does.

In [ ]:
PLAY = False   # set to True to play

if PLAY:
    world = make_world()
    history = run_agent(GOAL, human_llm, human_consult, world, max_steps=8)

## Run 3: a real model

Same harness, same world, same goal: only the `llm` argument changes. Needs an `ANTHROPIC_API_KEY` (in Colab: the key icon in the left sidebar, then Secrets).

With `human_consult`, you're H: approve or refuse the model's risky actions, and watch whether it adapts.

In [ ]:
if get_api_key():
    world = make_world()
    history = run_agent(GOAL, claude_llm, human_consult, world, max_steps=8)
else:
    print("No ANTHROPIC_API_KEY found; skipping the real-model run.")

## What to take away

- **The agent is the loop, not the model.** Swapping a script, a human, and Claude in and out changed nothing else.
- **The model knows only what the prompt tells it.** No history in the prompt means no memory.
- **Capability = the tool portfolio.** Remove a tool and the action becomes impossible, not merely forbidden.
- **Oversight lives at the consult step**, and it's only as good as the description of the action that H is shown.
- **The harness judges completion here** because we could write `goal_achieved()`. For most real tasks we can't, and the model has to say when it's done.

## Exercises

1. **Let the model decide when it's done.** Add a `finish` tool and end the loop when it's called (instead of `goal_achieved`). How does the loop's shape change: is it still a `while`, or now a `repeat … until`?
2. **Shrink the senses.** Make `observe()` return nothing. What breaks, and what can the agent still learn from history alone?
3. **Forgetful agent.** Pass only the last two history entries into the prompt. Where does a real model go wrong?
4. **A misleading reason.** Write a script step whose `thought` says "reading the notes" but whose tool is `send_email`. Would a human reading only the thought catch it? Which part of `human_consult` protects you?
5. **An ethical oracle.** Replace `policy_consult` with a virtue oracle: the harness sends *situation + planned act + the virtue it claims to show*, and gets back `agree`, `wrong virtue`, `deficient`, or `excessive`. What does the harness have to compute before it can even ask?